## CIFAR-10 Dataset Preparation

CIFAR-10 consists of 32×32 RGB images across 10 classes.

For this experiment:
- We normalize pixel values to range [0,1].
- We reshape data to (batch, channels, height, width).
- We use only 500 training samples for faster training.

In [ ]:
import numpy as np
from tensorflow.keras.datasets import cifar10

np.random.seed(42)

# Load CIFAR-10
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# Normalize
X_train = X_train.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0

# Reshape to (batch, channels, height, width)
X_train = np.transpose(X_train, (0, 3, 1, 2))
X_test = np.transpose(X_test, (0, 3, 1, 2))

# Flatten labels
y_train = y_train.flatten()
y_test = y_test.flatten()

# Reduce dataset
X_train_small = X_train[:500]
y_train_small = y_train[:500]

X_test_small = X_test[:200]
y_test_small = y_test[:200]

print("Train shape:", X_train_small.shape)
print("Test shape:", X_test_small.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 11s 0us/step
Train shape: (500, 3, 32, 32)
Test shape: (200, 3, 32, 32)


## Conv2D Layer

The Convolution layer extracts spatial features from input images.

It:
- Uses learnable filters (kernels)
- Slides them across the image
- Performs element-wise multiplication
- Produces feature maps

Backward pass computes gradients for:
- Filter weights
- Bias
- Input tensor

In [ ]:
class Conv2D:
    def __init__(self, in_channels, out_channels, kernel_size):
        self.kernel_size = kernel_size
        self.out_channels = out_channels

        self.W = np.random.randn(out_channels, in_channels, kernel_size, kernel_size) \
                 * np.sqrt(2.0 / (in_channels * kernel_size * kernel_size))
        self.b = np.zeros((out_channels, 1))

        self.X = None

    def forward(self, X):
        self.X = X
        batch_size, in_channels, height, width = X.shape
        k = self.kernel_size

        out_h = height - k + 1
        out_w = width - k + 1

        out = np.zeros((batch_size, self.out_channels, out_h, out_w))

        for n in range(batch_size):
            for f in range(self.out_channels):
                for i in range(out_h):
                    for j in range(out_w):
                        region = X[n, :, i:i+k, j:j+k]
                        out[n, f, i, j] = np.sum(region * self.W[f]) + self.b[f, 0]

        return out

    def backward(self, d_out):
        batch_size, _, out_h, out_w = d_out.shape
        _, in_channels, height, width = self.X.shape
        k = self.kernel_size

        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)
        dX = np.zeros_like(self.X)

        for n in range(batch_size):
            for f in range(self.out_channels):
                for i in range(out_h):
                    for j in range(out_w):

                        region = self.X[n, :, i:i+k, j:j+k]

                        self.dW[f] += region * d_out[n, f, i, j]
                        self.db[f, 0] += d_out[n, f, i, j]
                        dX[n, :, i:i+k, j:j+k] += self.W[f] * d_out[n, f, i, j]

        return dX

## Quick shape test

In [ ]:
conv_test = Conv2D(3, 16, 3)
X_dummy = np.random.randn(2, 3, 32, 32)
out = conv_test.forward(X_dummy)

print("Conv output shape:", out.shape)

Conv output shape: (2, 16, 30, 30)


## ReLU Activation

ReLU (Rectified Linear Unit) introduces non-linearity.

It:
- Converts negative values to zero
- Keeps positive values unchanged

Backward pass:
- Gradient flows only where input was positive

In [ ]:
class ReLU:
    def __init__(self):
        self.X = None

    def forward(self, X):
        self.X = X
        return np.maximum(0, X)

    def backward(self, d_out):
        dX = d_out.copy()
        dX[self.X <= 0] = 0
        return dX

## MaxPooling Layer

MaxPooling reduces spatial dimensions.

It:
- Divides input into 2×2 regions
- Selects the maximum value from each region
- Reduces height and width by half

Backward pass:
- Gradient flows only to the position of the maximum value

In [ ]:
class MaxPool2D:
    def __init__(self, pool_size=2):
        self.pool_size = pool_size
        self.X = None
        self.mask = None

    def forward(self, X):
        self.X = X
        batch_size, channels, height, width = X.shape
        p = self.pool_size

        out_h = height // p
        out_w = width // p

        out = np.zeros((batch_size, channels, out_h, out_w))
        self.mask = np.zeros_like(X)

        for n in range(batch_size):
            for c in range(channels):
                for i in range(out_h):
                    for j in range(out_w):

                        region = X[n, c, i*p:(i+1)*p, j*p:(j+1)*p]
                        max_val = np.max(region)
                        out[n, c, i, j] = max_val

                        mask = (region == max_val)
                        self.mask[n, c, i*p:(i+1)*p, j*p:(j+1)*p] = mask

        return out

    def backward(self, d_out):
        batch_size, channels, out_h, out_w = d_out.shape
        p = self.pool_size

        dX = np.zeros_like(self.X)

        for n in range(batch_size):
            for c in range(channels):
                for i in range(out_h):
                    for j in range(out_w):

                        dX[n, c, i*p:(i+1)*p, j*p:(j+1)*p] += \
                            self.mask[n, c, i*p:(i+1)*p, j*p:(j+1)*p] * d_out[n, c, i, j]

        return dX

## CIFAR-10 CNN Architecture – Shape Planning

Input shape:
(batch_size, 3, 32, 32)

Layer-by-layer transformation:

- Conv1 (3 -> 16, kernel=3) Output:(batch_size, 16, 30, 30)

- MaxPool (2×2)
Output:
(batch_size, 16, 15, 15)

- Conv2 (16 -> 32, kernel=3)
Output:
(batch_size, 32, 13, 13)

- MaxPool (2×2)
Output:
(batch_size, 32, 6, 6)

- Flatten
32 × 6 × 6 = 1152 Output:
(batch_size, 1152)

- Fully Connected Layer
1152 -> 128

- Output Layer
128 -> 10 classes

- Final Output: (batch_size, 10)

## Fully Connected (Linear) Layer

The Linear layer performs:

Output = XW + b

It converts feature vectors into class scores.

Backward pass computes gradients for:
- Weights (W)
- Bias (b)
- Input (X)

In [ ]:
class Linear:
    def __init__(self, in_features, out_features):
        self.W = np.random.randn(in_features, out_features) * np.sqrt(2. / in_features)
        self.b = np.zeros((1, out_features))
        self.X = None

    def forward(self, X):
        self.X = X
        return X @ self.W + self.b

    def backward(self, d_out):
        self.dW = self.X.T @ d_out
        self.db = np.sum(d_out, axis=0, keepdims=True)
        return d_out @ self.W.T

## Softmax with Cross-Entropy Loss

- Softmax converts logits into probabilities.

- Cross-Entropy measures the difference between
predicted probabilities and true labels.

- Backward pass directly computes gradient with respect to logits.

In [ ]:
class SoftmaxCrossEntropy:
    def __init__(self):
        self.probs = None
        self.y_true = None

    def forward(self, logits, y_true):
        shifted = logits - np.max(logits, axis=1, keepdims=True)
        exp_scores = np.exp(shifted)
        self.probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

        self.y_true = y_true
        batch_size = logits.shape[0]

        loss = -np.log(self.probs[range(batch_size), y_true])
        return np.mean(loss)

    def backward(self):
        batch_size = self.probs.shape[0]
        d_logits = self.probs.copy()
        d_logits[range(batch_size), self.y_true] -= 1
        return d_logits / batch_size

## Adam Optimizer

Adam improves training by:

- Using momentum (first moment estimate)
- Using adaptive learning rates (second moment estimate)
- Applying bias correction

It updates parameters more efficiently than SGD.

In [ ]:
class Adam:
    def __init__(self, parameters, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.parameters = parameters
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon

        self.m = [np.zeros_like(p) for p in parameters]
        self.v = [np.zeros_like(p) for p in parameters]
        self.t = 0

    def step(self, grads):
        self.t += 1

        for i, (p, g) in enumerate(zip(self.parameters, grads)):
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * (g ** 2)

            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)

            p -= self.lr * m_hat / (np.sqrt(v_hat) + self.epsilon)

## Full CIFAR-10 CNN

**Architecture:**

Conv(3→16) -> ReLU -> Pool  
Conv(16→32) -> ReLU -> Pool  
Flatten  
FC(1152 -> 128) -> ReLU  
FC(128 -> 10)  
Softmax + CrossEntropy  
Adam Optimizer

We train using 500 samples for efficiency.

In [ ]:
# Initialize Layers
conv1 = Conv2D(3, 16, 3)
relu1 = ReLU()
pool1 = MaxPool2D(2)

conv2 = Conv2D(16, 32, 3)
relu2 = ReLU()
pool2 = MaxPool2D(2)

fc1 = Linear(32 * 6 * 6, 128)
relu3 = ReLU()

fc2 = Linear(128, 10)

loss_fn = SoftmaxCrossEntropy()

# Adam Optimizer
parameters = [
    conv1.W, conv1.b,
    conv2.W, conv2.b,
    fc1.W, fc1.b,
    fc2.W, fc2.b
]

optimizer = Adam(parameters, lr=0.001)

# Training Loop

In [ ]:
epochs = 10
batch_size = 50

for epoch in range(epochs):

    epoch_loss = 0

    for i in range(0, 500, batch_size):

        X_batch = X_train_small[i:i+batch_size]
        y_batch = y_train_small[i:i+batch_size]

        # -------- Forward --------
        out = conv1.forward(X_batch)
        out = relu1.forward(out)
        out = pool1.forward(out)

        out = conv2.forward(out)
        out = relu2.forward(out)
        out = pool2.forward(out)

        out_flat = out.reshape(out.shape[0], -1)

        out = fc1.forward(out_flat)
        out = relu3.forward(out)

        logits = fc2.forward(out)

        loss = loss_fn.forward(logits, y_batch)
        epoch_loss += loss

        # -------- Backward --------
        d_logits = loss_fn.backward()

        d_out = fc2.backward(d_logits)
        d_out = relu3.backward(d_out)
        d_out = fc1.backward(d_out)

        d_out = d_out.reshape(out_flat.shape[0], 32, 6, 6)

        d_out = pool2.backward(d_out)
        d_out = relu2.backward(d_out)
        d_out = conv2.backward(d_out)

        d_out = pool1.backward(d_out)
        d_out = relu1.backward(d_out)
        d_out = conv1.backward(d_out)

        # -------- Adam Update --------
        grads = [
            conv1.dW, conv1.db,
            conv2.dW, conv2.db,
            fc1.dW, fc1.db,
            fc2.dW, fc2.db
        ]

        optimizer.step(grads)

    print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}")

Epoch 1, Loss: 24.2372
Epoch 2, Loss: 22.3850
Epoch 3, Loss: 20.8152
Epoch 4, Loss: 18.9452
Epoch 5, Loss: 16.9313
Epoch 6, Loss: 15.6513
Epoch 7, Loss: 14.3704
Epoch 8, Loss: 12.6723
Epoch 9, Loss: 11.3728
Epoch 10, Loss: 10.4466


# Evaluation

In [ ]:
out = conv1.forward(X_test_small)
out = relu1.forward(out)
out = pool1.forward(out)

out = conv2.forward(out)
out = relu2.forward(out)
out = pool2.forward(out)

out_flat = out.reshape(out.shape[0], -1)

out = fc1.forward(out_flat)
out = relu3.forward(out)

logits = fc2.forward(out)

predictions = np.argmax(logits, axis=1)
accuracy = np.mean(predictions == y_test_small)

print("Test Accuracy:", accuracy)

Test Accuracy: 0.325


## CIFAR-10 Results

The CNN was trained on 500 CIFAR-10 images using Adam optimizer.

Results:
- Final Loss = 10.44
- Test Accuracy ≈= 32.5%

Observations:
- Loss decreased steadily, indicating correct gradient flow.
- Accuracy significantly exceeded random guessing (10%).
- CIFAR-10 is more complex than MNIST, explaining lower accuracy.
- The model successfully learned spatial features despite limited data.

This confirms the correctness of the CNN implementation on a more challenging dataset.

## Final Conclusion – CNN from Scratch (MNIST & CIFAR-10)

In this lab, we implemented a Convolutional Neural Network (CNN) entirely from scratch using NumPy, including:

- Conv2D (forward & backward propagation)
- MaxPooling (forward & backward propagation)
- ReLU activation
- Fully Connected layers
- Softmax with Cross-Entropy loss
- SGD optimizer
- Adam optimizer

The implementation was first implemented on MNIST and then on the more complex CIFAR-10 dataset.

---

### MNIST Results (1000 training samples)

SGD:
- Test Accuracy = 67.5%

Adam:
- Test Accuracy = 81%

Observations:
- Adam converged faster than SGD.
- The model successfully learned spatial digit features.
- MNIST is relatively simple, allowing high performance even with a small CNN.

---

### CIFAR-10 Results (500 training samples)

Adam:
- Final Loss = 10.44
- Test Accuracy = 32.5%

Observations:
- Loss decreased steadily, confirming correct backpropagation.
- Accuracy significantly exceeded random guessing (10%).
- CIFAR-10 is much more complex than MNIST, leading to lower accuracy.
- Limited training samples (500) restricted model generalization.

---

### Overall Insights

- Manual implementation deepened understanding of gradient flow in CNNs.
- Optimizer choice (Adam vs SGD) significantly affects convergence speed and accuracy.
- Dataset complexity plays a major role in achievable performance.
- Even a simple manually built CNN can learn meaningful visual features.

This lab validates the correctness and robustness of the CNN implementation across datasets of varying difficulty.